In [ ]:
from src.agent.prompts import (
    get_current_date,
    query_writer_instructions,
    web_searcher_instructions,
    reflection_instructions,
    answer_instructions,
)
from src.agent.state import (
    OverallState,
    QueryGenerationState,
    ReflectionState,
    WebSearchState,
)

In [ ]:

import dotenv
from rich import print as rprint
from src.agent.llm.llm import OpenAICompatibleLLM
from src.agent.base_agent import Agent
dotenv.load_dotenv(override=True)

model = 'MiniMax-M3'
deep_research_topic = '现在企业级的 AI Agent 落地方案深度调研'
initial_search_query_count = 2

## Agent class 说明

- set_step_prompt() ： 设置提示词

- step():
  - 使用 以上设置好的提示词，调用 LLM
  - 对结果进行后期处理（直接返回/json/解析异常等）
- __call__ ：llm 调用


In [ ]:

agent = Agent(model_id=model)
print('=============== prompt before format ===============')
rprint(query_writer_instructions)


agent.set_step_prompt(query_writer_instructions)
print('=============== prompt after format ===============')
prompt_formatted=agent.prompt_format(
    prompt=query_writer_instructions,
    current_date=get_current_date(),
    research_topic=deep_research_topic,
    number_queries=initial_search_query_count)
rprint(prompt_formatted)


response = agent.step(
  current_date=get_current_date(),
  research_topic=deep_research_topic,
  number_queries=initial_search_query_count,
)
rprint( 'response is str :',isinstance(response,str))
rprint(response)

## JsonAgent

- 指定一个 json 对象格式化的类
- 格式化输出
  - 去除 LLM 的 <think> 等标签
  - 正则方式 ： 只获取 ```{{ ... }} ``` 中的内容

### generate_search

In [17]:

from src.agent.base_agent import JsonAgent
from src.agent.tools_and_schemas import SearchQueryList, Reflection

agent = JsonAgent(model_id=model, keys=SearchQueryList)

init_prompt = query_writer_instructions
print('===============原始提示词， prompt before format ===============')
rprint(init_prompt)


agent.set_step_prompt(init_prompt)
print('===============提示词格式化， prompt after format ===============')
prompt_formatted=agent.prompt_format(
    prompt=init_prompt,
    current_date=get_current_date(),
    research_topic=deep_research_topic,
    number_queries=initial_search_query_count)
rprint(prompt_formatted)
print('=============== 调用 LLM，并 做 后期 post process 并返回结果 ===============')

result:SearchQueryList = agent.step(
    current_date=get_current_date(),
    research_topic=deep_research_topic,
    number_queries=initial_search_query_count,
)
rprint( 'result is SearchQueryList :',isinstance(result,SearchQueryList), type(result))

rprint(result)

===============原始提示词， prompt before format ===============


# 任务说明
你的任务是根据当前的研究主题决定多个用于网络搜索的标题，这些标题会被用于从网页搜集信息，并整合成一份专业的研究报告

# Instruction
- 针对当前的研究主题，你可以将其拆解成若干个搜索主题，每个搜索主题都应该是针对当前研究主题不同维度的切分
- 针对当前研究主题，最多不产生{number_queries}条搜索主题
- 你的搜索主题应该尽可能的广泛，如果研究主题本身就非常宽泛，则产出1条以上的搜索主题
- 每个搜索主题应该具备独立性，即不要同时产出多个相似或者耦合的搜索主题
- 搜索主题应该考虑时间，即除非研究主题要求，不然尽可能搜集近期的资料，当前时间是{current_date}

# Output Format
你生成的内容应该是一个标准的json格式的内容，并包含两个字端
<param>
 <attribute>rationale</attribute>
 <type>string</type>
 <description>你的思考，即为什么要产出如下的几个搜索主题</description>
</param>

<param>
 <attribute>query</attribute>
 <type>List</type>
 <description>用于做网络搜索的搜索主题</description>
</param>
下面是一个输出样例
```json
{
 "rationale": "xxxx",
 "query": ["搜索主题1", "搜索主题2", ...]
}
```

# Context
{research_topic}

# Output

===============提示词格式化， prompt after format ===============


# 任务说明
你的任务是根据当前的研究主题决定多个用于网络搜索的标题，这些标题会被用于从网页搜集信息，并整合成一份专业的研究报告

# Instruction
- 针对当前的研究主题，你可以将其拆解成若干个搜索主题，每个搜索主题都应该是针对当前研究主题不同维度的切分
- 针对当前研究主题，最多不产生2条搜索主题
- 你的搜索主题应该尽可能的广泛，如果研究主题本身就非常宽泛，则产出1条以上的搜索主题
- 每个搜索主题应该具备独立性，即不要同时产出多个相似或者耦合的搜索主题
- 搜索主题应该考虑时间，即除非研究主题要求，不然尽可能搜集近期的资料，当前时间是September 02, 2026

# Output Format
你生成的内容应该是一个标准的json格式的内容，并包含两个字端
<param>
 <attribute>rationale</attribute>
 <type>string</type>
 <description>你的思考，即为什么要产出如下的几个搜索主题</description>
</param>

<param>
 <attribute>query</attribute>
 <type>List</type>
 <description>用于做网络搜索的搜索主题</description>
</param>
下面是一个输出样例
```json
{
 "rationale": "xxxx",
 "query": ["搜索主题1", "搜索主题2", ...]
}
```

# Context
现在企业级的 AI Agent 落地方案深度调研

# Output

=============== 调用 LLM，并 做 后期 post process 并返回结果 ===============


response is SearchQueryList : True <class 'src.agent.tools_and_schemas.SearchQueryList'>

SearchQueryList(
    query=[
        '2025 2026 企业级 AI Agent 框架平台对比 LangChain AutoGen CrewAI Salesforce Agentforce Microsoft Copilot 
Studio 落地趋势',
        '企业 AI Agent 落地案例 架构设计 部署实践 挑战与经验 金融 制造 客服场景'
    ],
    rationale="企业级 AI Agent 
落地是一个涉及技术架构、框架平台、行业实践、挑战与最佳实践等多个维度的复杂课题。为了深度调研，需要从'市场全景与技术
选型'和'企业实际落地案例与架构经验'两个互补的角度切入：前者帮助理解当前主流的 Agent 
框架、平台和厂商格局（LangChain、AutoGen、CrewAI、Salesforce Agentforce、Microsoft Copilot Studio 等）以及 
2025-2026 
年的最新演进趋势；后者聚焦真实部署案例、架构设计、踩坑教训，更贴近'落地'这一核心诉求。两条搜索主题彼此独立，覆盖了'
看什么'和'怎么用'两大关键维度。"
)

### send_to_web_search

In [23]:
from agent.graph import WEB_SEARCH_NODE
from langgraph.types import Send
state = {
    'search_query':result.query
}
sends= [
    Send(WEB_SEARCH_NODE, {"search_query": search_query, "id": int(idx)})
    for idx, search_query in enumerate(state["search_query"])
]

rprint(sends)

[
    Send(node='web_search', arg={'search_query': '2025 2026 企业级 AI Agent 框架平台对比 LangChain AutoGen CrewAI 
Salesforce Agentforce Microsoft Copilot Studio 落地趋势', 'id': 0}),
    Send(node='web_search', arg={'search_query': '企业 AI Agent 落地案例 架构设计 部署实践 挑战与经验 金融 制造 
客服场景', 'id': 1})
]

### web_search



In [27]:
from src.agent.base_agent import WebSearchAgent

dotenv.load_dotenv(override=True)
for send in sends:
    arg = send.arg
    rprint(arg)
    web_searcher = WebSearchAgent()

    # 执行搜索
    response = web_searcher.step(prompt=arg["search_query"],
                                 count=3)
    rprint(response)

{
    'search_query': '2025 2026 企业级 AI Agent 框架平台对比 LangChain AutoGen CrewAI Salesforce Agentforce 
Microsoft Copilot Studio 落地趋势',
    'id': 0
}

2026-09-02 12:30:00.512 | INFO     | src.agent.base_agent:__init__:29 - 速率限制器已初始化: 最大QPS=12.0, 最小间隔=0.083秒


[
    {
        'snippet': 'Coworker AI 2026年的报告指出,最佳替代方案包括:**LlamaIndex、Haystack、CrewAI、Microsoft 
Semantic Kernel、AutoGen**,以及面向团队的托管平台如Coworker本身。选择标准取决于对控制粒度与交付速度的权衡。 ## 2. 
技术原理与架构对比 ### 2.1 按场景分类 | 场景 | 推荐框架 | 核心优势 | |------|----------|----------| | 
检索增强生成(RAG) | LlamaIndex, Haystack | 原生文档索引、分块策略、检索器优化 | |多Agent系统 | CrewAI, AutoGen | 
角色分工、任务编排、对话管理 | | 企业级集成与治理 | Coworker Platform | 预置40+企业应用连接器,自动映射关系 | | 
微软生态 | Semantic Kernel | 与Azure OpenAI、Copilot深度集成 | ### 2.2 关键技术差异 - 
**LlamaIndex**(最新稳定版0.12.x):以“索引”为核心,提供`VectorStoreIndex`、`SummaryIndex`等,支持自定义检索器重排序。其
`QueryEngine`一次性封装了检索+生成,性能优于LangChain的`RetrievalQA`。 - 
**Haystack**(2.9+):采用`Pipeline`架构,组件化程度高,支持`document_store`、`retriever`、`reader`等独立模块,且内置`Ela
sticsearch`、`Weaviate`等生产级后端。其`haystack-ai`库在2025年通过了CNCF的初步评估。 - 
**CrewAI**(0.30+):基于“角色-任务-流程”模型,每个Agent有独立`role`、`goal`、`backstory`,通过`Task`和`Crew`协作。支持`
sequential`、`hierarchical`等流程,内置`AgentExecutor`处理工具调用。 - 
**AutoGen**(0.7+):微软开源,强调“对话式多Agent”,通过`ConversableAgent`和`GroupChat`实现agent间多轮交互。支持`code_ex
ecution`和`human_input_mode`,适合复杂推理任务。',
        'title': 'LangChain退场?2026年五大替代框架深度对比',
        'url': 'https://blog.csdn.net/m0_69581581/article/details/163166782'
    },
    {
        'snippet': '一、为什么2026年还需要做框架对比? 
如果你过去半年参加过任何一场AI工程相关的技术分享,大概率会听到这么一句话: 
“2024年选框架看Star数,2026年选框架看Checkpoint能不能从断点恢复。” 
这不是段子。AIAgent开发已经从"能不能跑Demo"进入了"工程化落地"阶段。去年大家讨论的是"Agent能不能调工具",今年讨论的是
"93个Agent同时跑,状态怎么合并、故障怎么恢复、Token成本怎么控制"。 
一个真实的场景:某个中型团队花了3周用某框架搭了一套客服Agent系统,Demo效果惊艳,老板拍板上线。结果第一周就遇到了长对话
状态丢失、多Agent并行时死循环、人工接管流程缺失三大问题。最后重构花了整整两个月。 
选错框架的代价,不是换个包的问题,是40+小时的返工成本和整个团队对AI方向的信心消耗。 
本文选取了2026年最具代表性的4个框架——LangChain(含LangGraph)、AutoGen、CrewAI、Dify,从一线开发者的视角,从5个核心维度
做深度对比,并给出4个典型企业场景的选型建议。 提前声明:没有"最好"的框架,只有"最适合你当前阶段"的框架。 
二、四大框架核心特点速览 先看全局,再看细节。 
<table><tr><th>维度</th><th>LangChain/LangGraph</th><th>AutoGen</th><th>CrewAI</th><th>Dify</th></tr><tr><td>定位</
td><td>全栈Agent开发框架 + 
状态图编排引擎</td><td>多Agent对话协作框架</td><td>角色化多Agent协作框架</td><td>可视化LLMOps平台</td></tr><tr><td>
核心哲学</td><td>有向图状态机,你画图它执行</td><td>Agent通过结构化对话解决问题</td><td>定义角色、分配任务、启动协作
</td><td>拖拽画布搭建AI应用</td></tr><tr><td>2026最新版本</td><td>LangChain 1.3 / LangGraph 2.0</td><td>v0.4(向MS 
Agent Framework过渡中)</td><td>持续迭代,企业版发力中</td><td>2.0+(原生MCP支持)</td></tr><tr><td>GitHub 
Stars</td><td>LangChain ~100K+ / LangGraph 
~97K</td><td>~38K(原仓库)</td><td>~47K</td><td>~145K</td></tr><tr><td>开发语言</td><td>Python / 
JS/TS</td><td>Python</td><td>Python</td><td>Python后端 + 
Next.js前端</td></tr><tr><td>上手门槛</td><td>中等偏高</td><td>中等</td><td>低(3行代码出原型)</td><td>低(可视化拖拽
)</td></tr><tr><td>生产就绪度</td><td>高(配合LangSmith)',
        'title': '企业级AI Agent开发:4大主流框架深度对比与选型指南(2026版)',
        'url': 'https://blog.csdn.net/hicaishen/article/details/162912714'
    },
    {
        'snippet': '主流Agent 框架对比 
目前开源Agent框架百花齐放,但95%的落地场景,都集中在三大主流框架,各自定位清晰、各司其职。 LangChain:AI开发界的Spring 
Boot 核心定位:企业级定制、全场景通用开发框架 LangChain 
生态最完善、组件最丰富,支持数百种模型与工具集成,灵活性拉满。适合开发者自主搭建定制化Agent,是工业级落地的首选框架。
缺点是需要自主组装组件,纯新手快速上手难度略高。 AutoGPT:全自动自主智能体 核心定位:零配置、目标驱动的全自动Agent 
无需复杂开发,只需输入最终目标,即可自主完成规划、检索、执行、复盘。适合快速验证想法、开放式调研任务,但高度定制化的企
业业务场景适配性较弱。 CrewAI+AutoGen:多Agent协作集群 核心定位:多智能体分工协作,模拟团队办公 
打破单Agent能力局限,支持角色拆分、任务分配、消息互通、结果汇总。',
        'title': '彻底搞懂AI Agent:从原理、主流框架到到实战落地',
        'url': 'https://cloud.tencent.com/developer/article/2696896'
    },
    {
        'snippet': '二、框架分层逻辑:先理解 Agent 系统的三层结构 三、LangGraph:状态机驱动的精密仪器 3.1 核心架构 
3.2 适用场景 四、CrewAI:角色扮演的特种部队 生成一个适合你的列表 创建一个表格 设定内容居中、居左、居右 SmartyPants 
创建一个自定义列表 如何创建一个注脚 导入 一、Agent 工程的时代背景 2026 年,AI Agent 
工程正经历从"实验室玩具"到"生产基础设施"的关键跨越。行业调研数据显示:78% 的企业已启动 AI Agent 试点项目,但只有 14% 
成功跨越了从试点到生产规模的鸿沟,而框架选型错误是导致失败的首要原因,占比高达 43%。 这个数据说明:Agent 
能不能落地,很大程度上取决于你选对了框架、用对了方法。市面上的 Agent 
框架五花八门,各有各的适用场景。这篇文章不吹不黑,从架构理念、核心特性、适用场景三个维度,把主流框架拆开讲清楚,帮你做
出理性的选型决策。 在对比框架之前,先建立一个整体认知。Agent 系统架构可以分为三个层次: 
一是工具层。提供基础能力,包括检索、工具调用、记忆等。这一层与具体框架关系不大,更多是生态问题——框架能接入多少工具、
支持哪些协议(如 MCP)。 二是编排层。负责 Agent 
流程控制与协调,这是框架的核心价值所在,也是本文对比的重点。编排层决定了"Agent 
怎么思考、怎么决策、怎么调用工具、怎么处理多轮循环"。 三是应用层。面向特定场景的高层抽象,比如客服 Agent、编程 
Agent、数据分析 Agent 等

{'search_query': '企业 AI Agent 落地案例 架构设计 部署实践 挑战与经验 金融 制造 客服场景', 'id': 1}

[
    {
        'snippet': '2. 企业级AI Agent的四大落地场景解析 2.1 智能客服中心的"静默革命" 
传统客服系统转型是最典型的切入点。某银行信用卡中心部署的Agent系统,通过以下架构实现服务升级: 
语音接口层:采用基于Wav2Vec 2.0的语音识别模块,方言识别准确率达92% 
业务逻辑层:嵌套138个业务决策节点,覆盖86%的常见咨询场景 知识库层:动态更新的金融法规库和产品手册,变更同步延迟<30分钟 
实测数据显示,平均通话时长从4分12秒缩短至2分08秒,同时客户满意度提升11个百分点。关键在于设计了"人工接管度"指标——当Age
nt置信度低于85%时自动转人工,既保证效率又控制风险。 2.2 生产线的"数字老师傅" 
制造业的设备运维Agent正在改写传统巡检模式。某汽车电池工厂的案例值得参考: 
知识沉淀阶段:采集3位资深工程师的2000+条诊断经验 模型训练阶段:结合设备传感器数据构建故障特征库 
部署优化阶段:通过AR眼镜实现"所见即诊断"的增强现实指导 这套系统将新员工培',
        'title': '企业级AI Agent落地实战:场景解析与实施指南',
        'url': 'https://blog.csdn.net/anwenzhao0749/article/details/101631935'
    },
    {
        'snippet': 
'摘要/引言要么是通用Agent太「飘」,输出内容幻觉多、不符合行业规范,根本不敢用在生产环境;要么是定制化开发成本高,动辄上
百万的投入,还不知道能不能拿到预期收益。 本文我会把自己过去2年带队在金融、医疗、制造三个ToB核心领域落地AI 
Agent的真实经验全部拆解给你:从垂直行业Agent的核心设计逻辑,到三个领域的完整架构、核心代码、效果验证、ROI测算,再到我
踩过的30+个坑的避坑指南。读完本文你不仅能搞懂AI 
Agent在垂直行业的落地逻辑,还能拿到可直接复用的开发框架,哪怕你是第一次做Agent落地,也能避开80%的常见问题。 
本文的组织结构如下:先讲AI 
Agent的核心概念和垂直领域的特殊要求,再分别拆解金融、医疗、制造三个领域的落地实践,之后讲性能优化、常见问题和未来趋势
,最后给大家提供可复用的代码仓库。 想要落地AI Agent的企业数字化负责人、产品经理 
有AI基础,想要转向垂直领域Agent开发的后端/算法工程师 关注AI行业落地的投资人、行业分析师 前置知识 
了解大语言模型的基本常识,知道Prompt工程、RAG的基本概念 有基础的Python开发能力,能看懂简单的API调用代码 
对ToB行业的业务逻辑有基本认知 核心概念与理论基础:垂直行业Agent的设计逻辑 环境准备:通用Agent开发栈一键配置 
分步实现1:金融领域智能投研Agent落地 分步实现2:医疗领域基层临床辅助决策Agent落地 
分步实现3:制造领域设备预测性维护Agent落地 关键代码解析:三大领域通用的Agent核心模块设计 
结果展示与验证:三大领域落地的真实数据 性能优化与最佳实践 常见问题与解决方案 未来展望与扩展方向 为什么AI 
Agent值得关注? 
2024年国内大模型的推理成本已经降到了2022年的1%,7B参数开源模型的效果已经接近GPT-3.5的水平,AI落地的成本门槛已经被打穿
了。但传统的LLM应用(比如智能客服、文案生成)只能解决固定场景的简单问题,面对复杂的、需要多步推理、需要对接内部系统的
场景根本无能为力: 
金融分析师要写一篇新能源行业研报,需要翻100+份财报、公告、新闻,传统LLM没有实时数据,也不会自动检索整理',
        'title': 'AI Agent行业应用案例:金融、医疗、制造领域的落地实践',
        'url': 'https://blog.csdn.net/2405_88636357/article/details/161150903'
    },
    {
        'snippet': '2026年,企业级AI 
Agent正从概念验证阶段迈向规模化落地阶段。据IDC数据显示,2025年中国企业级Agent市场规模已达190亿元,预计未来三年复合增
长率将超过110%。与此同时,全球57%的企业已在生产环境部署多步工作流AI Agent,其中大型企业应用比例达67%。 
然而,热潮之下亦存在“冰火两重天”的局面。行业调研数据显示,92%的企业已在核心业务中部署AI 
Agent,但仅23%实现规模化应用。73%的企业在部署AI 
Agent时面临“场景适配度不足”“技术栈整合困难”等核心挑战。这意味着大量企业仍处于试点阶段,尚未找到可复制的落地模式。 
在此背景下,制造、金融、电商、政务四大行业作为AI 
Agent率先发力的关键赛道,其方案成熟度与落地路径各有千秋。本文将基于行业调研与实践数据,系统分析四大领域的落地现状与演
进方向,为企业在AI Agent选型与部署中提供参考依据。 一、制造业:从“数据可视”走向“智能可执行” 制造业是AI 
Agent落地最早、场景最复杂的赛道之一。制造企业长期面临生产排期不精准、供应链协同不畅、设备维护滞后等痛点。传统制造系
统以ERP、MES等为代表,本质上是“流程型”和“记录型”系统,依赖静态规则,缺乏敏捷响应能力。 当前,AI 
Agent在制造业的落地已覆盖研发设计、生产优化、供应链协同、设备运维等关键环节。在供应链管理方面,AI 
Agent通过实时分析订单数据、生产数据、库存数据及供应商表现,结合定制AI模型动态计算策略,实现科学、主动的库存管理。在生
产制造环节,产线运营智能体能够实时分析生产数据、市场需求和供应链情况,为管理者提供精准的生产排期建议,通过多维度数据融
合分析动态给出产线运行节奏,有效降低库存成本、提升设备利用效率。 
值得关注的是,工业企业中探索智能体的比例正在显著提升。IDC调研数据显示,已应用大模型及智能体的工业企业比例已从2024年的
9.6%提升至2025年的47.5%,其中在多个环节开展应用的企业从1.7%提升至35%。 制造业AI 
Agent方案的成熟度呈现以下特征:在供应链优化、库存管理等具备明确数据基础和业务规则的场景中落地效果较为成熟;在复杂工艺
优化、跨系统协同等场景中仍需持续迭代。总体来看,制造业AI 
Agent已从“单点试验”走向“多环节部署”,但距离全链条智能化仍有较大空间。 
金融行业作为数据密集型领域,长期面临数据孤岛、流程僵化、合规与效率平衡三大核心挑战。',
        'title': '企业级AI Agent落地标杆案例:制造/金融/电商/政务,哪家方案最成熟?',
        'url': 'https://www.shushangyun.com/article-36952.html'
    },
    {
        'snippet': 
'无论是制造、金融,还是政企服务,几乎所有行业都在探索如何用智能体实现流程自动化、知识问答和决策辅助。 
然而,很多企业的兴奋并没有持续太久。试点项目效果不稳定,成本一再飙升,安全部门也开始焦虑数据是否被“带走”。 
据公开数据显示,超过66%的企业将“结果可靠性”视为AI Agent落地的最大挑战。换句话说,AI Agent虽然火爆,但“坑”也不少。 
本文将结合行业报告与多家标杆企业的实践案例,系统拆解企业在AI 
Agent落地过程中面临的三大核心挑战,并提供一套经过验证的解决方案与落地建议,帮助你在浪潮中稳步前行。 1.1 问题表现 在AI
Agent落地的初期,大部分企业最先遇到的就是“输出不准、结果不稳”的问题: 
输出“幻觉”:模型“信口开河”,虚构法规、编造数据。例如在财务场景中,它可能生成一条根本不存在的税收政策,甚至伪造公告号。 
行为“失控”:同样的输入在不同时间得到不同答案,模型表现时好时坏,难以复现结果。 
逻辑混乱:在执行多步骤任务(如流程审批或报告生成)时,Agent可能在中途丢失上下文,导致任务失败。 
这些问题看似“智能不够”,实则源自系统性机制问题。如果不解决,AI Agent将难以支撑关键业务。 大模型的概率生成机制: 
LLM(大语言模型)并不是在“理解事实”,而是在根据概率预测最可能的下一个词。它并非基于逻辑推理,而是基于统计模式生成,因此
天然存在“幻觉风险”。 上下文窗口限制: 
模型一次能读取的上下文信息有限。当任务涉及多轮交互或大文档时,早先的关键信息被遗忘,导致回答前后矛盾。 
缺乏专业知识与语义边界: 训练数据中行业知识稀薄,例如财务、建筑、法律等领域,模型“懂点皮毛”,但不具备严谨的专业逻辑。 
提示词敏感性高: 同一个问题,稍改语气、顺序或关键词,答案就完全不同。这种“提示词漂移”让企业在生产中难以保障一致性。 
1.3 解决方案与最佳实践 ✅ 方案一:大小模型协同作战——以分工机制稳定结果 做法: 
将系统拆分为“大模型负责理解,小模型负责执行”的架构: 大模型(如GPT、Claude)擅长语义理解、任务规划、总结归纳; 
小模型(如轻量分类模型、规则算法)专注于具体计算、判断或检索。 这种分层机制可显著提升稳定性与一致性。 实践案例: 
联想在构建“端侧智能体”时采用了大模型+小模型+规则引擎的混合架构。',
        'title': 'AI Agent